# Featherweight — Phase 3 Group C: base Llama-3.1-8B BFCL baseline (Colab T4)

Runs the **base** model through the real BFCL harness under the *same*
`SalesforceLlamaHandler` prompting contract our fine-tune was trained on, so the
eventual base-vs-FT delta is apples-to-apples. GPT-4o was already done on the Mac.

**Approach (resolved from pinned bfcl-eval 2026.3.23 source):**
- We launch our **own** quantized vLLM server (bfcl's built-in launch can't pass
  `--quantization`), then point bfcl at it with `--skip-server-setup`.
- `scripts/run_bfcl.py` registers a custom `featherweight-base` entry
  (`SalesforceLlamaHandler`, `is_fc_model=False` => prompting path) then runs the
  bfcl CLI in-process.

> **Runtime risk:** bnb-4bit + vLLM on a T4 (Turing/SM 75) is the least-trodden
> path. If the server won't start, see the fallback note in the serve cell.


## 1. GPU + install

In [ ]:
!nvidia-smi

In [ ]:
# vLLM (server) + bfcl-eval (harness) + soundfile (transitive dep bfcl doesn't pin)
!pip install -q vllm 'bfcl-eval==2026.3.23' soundfile

## 2. Repo + package

In [ ]:
import os
REPO = '/content/Featherweight'
![ -d {REPO}/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight {REPO}
%cd {REPO}
!pip install -q -e .
import sys; sys.path.insert(0, f'{REPO}/src')  # editable .pth is read only at startup
from featherweight import config
print('ROOT_DIR:', config.ROOT_DIR)
CATS = ','.join(config.CONFIG.eval.categories)
print('categories:', CATS)

## 3. BFCL project root + serving env
`BFCL_PROJECT_ROOT` is where bfcl writes `result/`+`score/` (kept under the repo).

In [ ]:
import os
BASE_ID = config.BASE_MODEL_4BIT  # unsloth/llama-3.1-8b-Instruct-bnb-4bit (ungated)
os.environ['BFCL_PROJECT_ROOT'] = f'{REPO}/third_party/bfcl'
os.environ['LOCAL_SERVER_ENDPOINT'] = 'localhost'
os.environ['LOCAL_SERVER_PORT'] = '8000'
os.makedirs(os.environ['BFCL_PROJECT_ROOT'], exist_ok=True)
print('BFCL_PROJECT_ROOT =', os.environ['BFCL_PROJECT_ROOT'])
print('serving base id   =', BASE_ID)

## 4. Launch our own quantized vLLM server (background)
Serves the bnb-4bit base under its real id (so bfcl's request `model` + tokenizer
match). `--dtype half` because the T4 has no bf16.

**Fallback if this fails:** drop `--load-format bitsandbytes` (for some vLLM
versions `--quantization bitsandbytes` alone suffices), lower `--max-model-len`,
or serve `meta-llama/Llama-3.1-8B-Instruct` with `--quantization bitsandbytes`
(needs an HF token; gated).

In [ ]:
import subprocess, time, requests
server = subprocess.Popen([
    'vllm', 'serve', BASE_ID,
    '--quantization', 'bitsandbytes', '--load-format', 'bitsandbytes',
    '--dtype', 'half', '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.9', '--port', '8000',
])
# wait until /v1/models responds (model loaded)
for _ in range(120):
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).status_code == 200:
            print('vLLM server ready'); break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('vLLM server did not become ready - check output above')

## 5. Generate (base, skip-server-setup -> our server)

In [ ]:
!python scripts/run_bfcl.py generate --model featherweight-base \
    --backend vllm --skip-server-setup --test-category {CATS}

## 6. Evaluate (CPU scoring)

In [ ]:
!python scripts/run_bfcl.py evaluate --model featherweight-base --test-category {CATS}

## 7. Parse scores + stop the server
Copy the score dir out (download / Drive / HF) so the local report can combine it
with the GPT-4o baseline via `report.write_baselines`.

In [ ]:
from pathlib import Path
from featherweight.eval import report
score_dir = Path(os.environ['BFCL_PROJECT_ROOT']) / 'score' / 'featherweight-base'
for cat, s in report.collect_scores(score_dir).items():
    print(cat, s)

# zip the score dir for download
import shutil
shutil.make_archive('/content/base_scores', 'zip', score_dir)
print('scores zipped -> /content/base_scores.zip')
server.terminate()